# Encoding: como colunas de texto viram número

Modelo não multiplica palavra. Antes de treinar, toda coluna de texto precisa virar número, e a
escolha de **como** fazer isso muda o resultado mais do que a escolha do modelo em muitos problemas de
tabela com categórica.

São duas perguntas, nesta ordem:

1. **A ordem existe?** Se existe (`baixo`, `medio`, `alto`), a coluna vira número direto. Se não existe
   (`plano`, `cidade`), transformar em número inventa uma ordem.
2. **Quantas categorias?** Três categorias viram duas colunas. Quarenta viram trinta e nove colunas
   quase vazias, e aí a estratégia tem de mudar.

O notebook mede cada escolha na mesma validação cruzada e no mesmo modelo, com um alvo que depende, de
verdade, de duas categóricas e de uma terceira que só existe com efeito por cidade.


In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (OneHotEncoder, OrdinalEncoder, StandardScaler,
                                   TargetEncoder)

sns.set_theme(context="notebook", style="whitegrid", palette="colorblind",
              rc={"figure.figsize": (9, 4)})

RANDOM_STATE = 42
ALVO = "cancelou"


def tabela_com_categoricas(n_linhas=1200, n_cidades=40, seed=RANDOM_STATE):
    """Clientes com categórica ordinal, nominal, de cardinalidade alta e com faltante.

    O alvo depende de verdade de: plano (basico cancela mais), nivel_risco (ordem), um risco próprio
    de cada cidade, e as colunas numéricas. É esse risco por cidade que a codificação precisa capturar.
    `vendedor` entra de propósito SEM efeito nenhum: é o controle do experimento de vazamento.
    """
    rng = np.random.default_rng(seed)
    cidades = [f"cidade_{i:02d}" for i in range(1, n_cidades + 1)]
    risco_cidade = {cidade: rng.normal(0, 1.0) for cidade in cidades}

    tabela = pd.DataFrame({
        "idade": np.clip(rng.normal(38, 12, n_linhas), 18, 90).round().astype(int),
        "renda_mensal": rng.lognormal(mean=8.2, sigma=0.6, size=n_linhas).round(2),
        "chamadas_suporte": rng.poisson(1.4, n_linhas),
        "plano": rng.choice(["basico", "padrao", "premium"], n_linhas, p=[0.5, 0.35, 0.15]),
        "nivel_risco": rng.choice(["baixo", "medio", "alto"], n_linhas, p=[0.5, 0.3, 0.2]),
        "vendedor": rng.choice([f"vend_{i:04d}" for i in range(1, 601)], n_linhas),
        "cidade": rng.choice(cidades, n_linhas),
    })

    sinal = (0.90 * (tabela["plano"] == "basico")
             + 0.80 * (tabela["nivel_risco"] == "alto")
             + 0.40 * (tabela["nivel_risco"] == "medio")
             + tabela["cidade"].map(risco_cidade)
             - 0.0003 * tabela["renda_mensal"]
             + 0.35 * tabela["chamadas_suporte"]
             + rng.normal(0, 1.0, n_linhas))
    tabela[ALVO] = (sinal > np.quantile(sinal, 0.78)).astype(int)

    # faltante numérico e faltante em categórica, dois casos diferentes
    tabela.loc[rng.choice(n_linhas, size=60, replace=False), "renda_mensal"] = np.nan
    tabela.loc[rng.choice(n_linhas, size=48, replace=False), "plano"] = np.nan
    return tabela


tabela = tabela_com_categoricas()
NUMERICAS = ["idade", "renda_mensal", "chamadas_suporte"]
ORDINAL = ["nivel_risco"]
NOMINAIS = ["plano"]
ALTA_CARDINALIDADE = ["cidade"]

print(f"tabela: {tabela.shape[0]} linhas x {tabela.shape[1]} colunas")
print(f"alvo: {tabela[ALVO].mean():.1%} de cancelamento")
print()
print(tabela.head(5).to_string())


## O que cada coluna de texto é

Cardinalidade e ordem são as duas informações que decidem a estratégia. O desenho vem antes da conta.

A tabela tem ainda `vendedor`, com centenas de categorias e nenhuma relação com o alvo. Ele está ali de
propósito: é o controle do experimento de vazamento, no fim do notebook.


In [ ]:
categoricas = ORDINAL + NOMINAIS + ALTA_CARDINALIDADE + ["vendedor"]
resumo = pd.DataFrame({
    "categorias": tabela[categoricas].nunique(),
    "tem ordem": ["sim (baixo, medio, alto)", "nao", "nao", "nao"],
    "faltante (%)": (tabela[categoricas].isna().mean() * 100).round(2).values,
})
display(resumo.set_index(pd.Index(categoricas, name="coluna")))

fig, eixos = plt.subplots(1, 2, figsize=(13, 3.4))
sns.countplot(tabela, x="plano", order=tabela["plano"].value_counts().index, ax=eixos[0])
eixos[0].set_title("plano: 3 categorias, nominal")
sns.countplot(tabela, x="nivel_risco", order=["baixo", "medio", "alto"], ax=eixos[1])
eixos[1].set_title("nivel_risco: 3 categorias, com ordem")
plt.tight_layout()
plt.show()

frequencia_cidade = tabela["cidade"].value_counts()
sns.barplot(x=frequencia_cidade.head(12).values, y=frequencia_cidade.head(12).index)
plt.title("as 12 cidades mais frequentes (de 40)")
plt.xlabel("linhas")
plt.show()
print(f"cidade: {frequencia_cidade.size} categorias, "
      f"{frequencia_cidade.min()} a {frequencia_cidade.max()} linhas por categoria, "
      f"mediana {frequencia_cidade.median():.0f}")
print(f"categorias com menos de 30 linhas: {(frequencia_cidade < 30).sum()}")


## Quando a ordem existe: ordinal

`nivel_risco` tem ordem, e a ordem é informação: alto é mais grave que médio, que é mais grave que
baixo. Nesse caso o número é a resposta, e dizer qual é a ordem é parte do trabalho.


In [ ]:
ordinal = OrdinalEncoder(categories=[["baixo", "medio", "alto"]],
                         handle_unknown="use_encoded_value", unknown_value=-1)
codificado = ordinal.fit_transform(tabela[ORDINAL])
print("baixo, medio, alto viram:", np.unique(codificado))
print()
display(pd.crosstab(tabela["nivel_risco"], tabela[ALVO], normalize="index").round(3))


O cruzamento mostra a ordem no efeito: quanto maior o risco, maior a taxa de cancelamento. Um número
que cresce junto com o efeito é exatamente o que um modelo linear aproveita.

O erro comum é aplicar a mesma ideia em `plano`, que **não** tem ordem. O modelo recebe 0, 1 e 2 e
passa a ler "premium é o dobro de basico", o que não existe. A célula da comparação mede o preço desse
erro.


## Quando a ordem não existe: one-hot

One-hot cria uma coluna por categoria, com 1 na categoria da linha e 0 nas outras. Para `plano`, três
categorias viram três colunas (ou duas, se você tirar uma, como fazem os modelos lineares para não
deixar uma coluna ser soma das outras). O valor faltante conta como categoria própria: `plano` tem
três valores e um faltante, e o one-hot cria quatro colunas para ele.


In [ ]:
um_calor = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
matriz = um_calor.fit_transform(tabela[NOMINAIS + ["nivel_risco"]])
print(f"antes: {tabela[NOMINAIS + ['nivel_risco']].shape[1]} colunas de texto")
print(f"depois do one-hot: {matriz.shape[1]} colunas numericas")
print("as colunas que nasceram:", list(um_calor.get_feature_names_out()))
print(f"proporcao de zeros na matriz: {(matriz == 0).mean():.1%}")

# categoria nova, que aparece so na hora de prever
novo = pd.DataFrame({"plano": ["empresarial"], "nivel_risco": ["alto"]})
print()
print("com handle_unknown='ignore' e categoria nova:", um_calor.transform(novo))


`handle_unknown="ignore"` é o que evita o notebook que funciona no seu computador e quebra em produção
na primeira categoria nova: em vez de erro, a linha vira zeros naquela família de colunas.

E aí chega o problema de `cidade`: quarenta categorias viram quarenta colunas, cada uma com cerca de
vinte e três casos. O modelo passa a estimar um efeito por cidade em cima de vinte linhas, o que é
quase ruído. Três saídas possíveis, medidas abaixo:

1. one-hot mesmo assim, e aceitar o custo;
2. juntar as categorias raras num único rótulo `outras`, decisão que **não usa o alvo** e pode ser
   tomada antes da divisão;
3. codificar pela relação com o alvo, que é o `TargetEncoder`, e é a única das três que usa o alvo.


In [ ]:
def agrupar_raras(serie, minimo=30, rotulo="outras"):
    """Junta num rotulo so as categorias com menos de `minimo` linhas. Nao usa o alvo."""
    frequencia = serie.value_counts()
    return serie.map(lambda valor: valor if frequencia[valor] >= minimo else rotulo)


cidade_agrupada = agrupar_raras(tabela["cidade"], minimo=30)
print(f"cidade: {tabela['cidade'].nunique()} categorias -> {cidade_agrupada.nunique()}")
print(cidade_agrupada.value_counts().head(5).to_string())


## Codificar pela relação com o alvo, e a armadilha

`TargetEncoder` troca cada categoria pela média do alvo naquela categoria. Ele resolve cardinalidade
alta, porque não cria coluna nenhuma: a cidade vira um número só. E traz dois cuidados:

- **suavização** (`smooth`): categoria com vinte casos tem média instável, então o valor é puxado na
  direção da média geral;
- **vazamento**: se a média for calculada usando a própria linha que está sendo codificada, o modelo vê
  o alvo. A implementação do scikit-learn resolve isso com validação cruzada interna, mas ela só tem
  como saber se você deixar o codificador **dentro** do pipeline, para ele ser ajustado a cada fold.


In [ ]:
def auc_medio(pipeline, X, y, folds=5):
    cv = StratifiedKFold(folds, shuffle=True, random_state=RANDOM_STATE)
    notas = cross_val_score(pipeline, X, y, cv=cv, scoring="roc_auc")
    return notas.mean(), notas.std()


def monta(codificador_cidade=None, codificador_nominal=None):
    """ColumnTransformer com imputacao, escala nas numericas e o que for pedido nas categoricas."""
    passo_nominal = codificador_nominal if codificador_nominal is not None else         OneHotEncoder(handle_unknown="ignore")
    transformadores = [
        ("num", Pipeline([("imputa", SimpleImputer(strategy="median")), ("escala", StandardScaler())]),
         NUMERICAS),
        ("ordinal", OrdinalEncoder(categories=[["baixo", "medio", "alto"]],
                                   handle_unknown="use_encoded_value", unknown_value=-1), ORDINAL),
        ("nominal", passo_nominal, NOMINAIS),
    ]
    if codificador_cidade is not None:
        transformadores.append(("cidade", codificador_cidade, ALTA_CARDINALIDADE))
    return ColumnTransformer(transformadores, remainder="drop")


modelo = LogisticRegression(max_iter=5000)
X = tabela.drop(columns=[ALVO])
y = tabela[ALVO]

linhas = []

# 1. so as numericas: o piso, sem nenhuma categórica
baseline = Pipeline([("imputa", SimpleImputer(strategy="median")), ("escala", StandardScaler()),
                     ("modelo", LogisticRegression(max_iter=5000))])
media, desvio = auc_medio(baseline, X[NUMERICAS], y)
linhas.append({"método": "so as numericas", "colunas": len(NUMERICAS),
               "roc_auc": media, "desvio": desvio})

# 2. ordinal em tudo, inclusive no nominal: o erro comum
errado = Pipeline([("codigos", ColumnTransformer([
    ("num", Pipeline([("imputa", SimpleImputer(strategy="median")), ("escala", StandardScaler())]),
     NUMERICAS),
    ("texto", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                             encoded_missing_value=-2), ORDINAL + NOMINAIS),
])), ("modelo", LogisticRegression(max_iter=5000))])
media, desvio = auc_medio(errado, X, y)
linhas.append({"método": "ordinal em tudo (plano vira 0,1,2)", "colunas": 5,
               "roc_auc": media, "desvio": desvio})

# 3. one-hot no nominal, cidade de fora
media, desvio = auc_medio(Pipeline([("codigos", monta()), ("modelo", modelo)]), X, y)
linhas.append({"método": "one-hot no plano, sem a cidade", "colunas": 6,
               "roc_auc": media, "desvio": desvio})

# 4. one-hot tambem na cidade
media, desvio = auc_medio(Pipeline([("codigos", monta(OneHotEncoder(handle_unknown="ignore"))),
                                    ("modelo", modelo)]), X, y)
linhas.append({"método": "one-hot tambem na cidade (40 colunas)", "colunas": 6 + 40,
               "roc_auc": media, "desvio": desvio})

# 5. cidade agrupada e depois one-hot
X_agrupado = X.copy()
X_agrupado["cidade"] = agrupar_raras(X_agrupado["cidade"], minimo=30)
media, desvio = auc_medio(Pipeline([("codigos", monta(OneHotEncoder(handle_unknown="ignore"))),
                                    ("modelo", modelo)]), X_agrupado, y)
linhas.append({"método": "cidade agrupada em raras + one-hot", "colunas": 6 + int(X_agrupado["cidade"].nunique()),
               "roc_auc": media, "desvio": desvio})

# 6. cidade pelo target encoder, dentro do pipeline (do jeito certo)
media, desvio = auc_medio(Pipeline([("codigos", monta(TargetEncoder(target_type="binary", cv=5,
                                                               random_state=RANDOM_STATE))),
                                    ("modelo", modelo)]), X, y)
linhas.append({"método": "target encoding na cidade (no pipeline)", "colunas": 6,
               "roc_auc": media, "desvio": desvio})

# 7. o mesmo target encoding, calculado antes da divisao: o vazamento
tabela_vazada = X.copy()
media_alvo = y.groupby(X["cidade"]).mean()
tabela_vazada["cidade"] = X["cidade"].map(media_alvo)
media_vazada, desvio_vazada = auc_medio(Pipeline([
    ("codigos", ColumnTransformer([
        ("num", Pipeline([("imputa", SimpleImputer(strategy="median")), ("escala", StandardScaler())]),
         NUMERICAS),
        ("ordinal", OrdinalEncoder(categories=[["baixo", "medio", "alto"]],
                                   handle_unknown="use_encoded_value", unknown_value=-1), ORDINAL),
        ("nominal", OneHotEncoder(handle_unknown="ignore"), NOMINAIS),
        ("cidade", "passthrough", ["cidade"]),
    ])), ("modelo", modelo)]), tabela_vazada, y)
linhas.append({"método": "target encoding calculado antes (vazado)", "colunas": 7,
               "roc_auc": media_vazada, "desvio": desvio_vazada})

# 8. controle do experimento: 600 vendedores, 2 linhas cada, sem nenhum efeito real no alvo
codigos_com_vendedor = ColumnTransformer([
    ("num", Pipeline([("imputa", SimpleImputer(strategy="median")), ("escala", StandardScaler())]),
     NUMERICAS),
    ("ordinal", OrdinalEncoder(categories=[["baixo", "medio", "alto"]],
                               handle_unknown="use_encoded_value", unknown_value=-1), ORDINAL),
    ("nominal", OneHotEncoder(handle_unknown="ignore"), NOMINAIS),
    ("cidade", TargetEncoder(target_type="binary", cv=5, random_state=RANDOM_STATE),
     ALTA_CARDINALIDADE),
    ("vendedor", TargetEncoder(target_type="binary", cv=5, random_state=RANDOM_STATE), ["vendedor"]),
])
media, desvio = auc_medio(Pipeline([("codigos", codigos_com_vendedor), ("modelo", modelo)]), X, y)
linhas.append({"método": "vendedor sem efeito, no pipeline", "colunas": 8,
               "roc_auc": media, "desvio": desvio})

# 9. o mesmo vendedor, com a media do alvo calculada em toda a base antes da divisao
tabela_vazada_vendedor = X.copy()
tabela_vazada_vendedor["vendedor"] = X["vendedor"].map(y.groupby(X["vendedor"]).mean())
media_ven, desvio_ven = auc_medio(Pipeline([
    ("codigos", ColumnTransformer([
        ("num", Pipeline([("imputa", SimpleImputer(strategy="median")), ("escala", StandardScaler())]),
         NUMERICAS),
        ("ordinal", OrdinalEncoder(categories=[["baixo", "medio", "alto"]],
                                   handle_unknown="use_encoded_value", unknown_value=-1), ORDINAL),
        ("nominal", OneHotEncoder(handle_unknown="ignore"), NOMINAIS),
        ("cidade", TargetEncoder(target_type="binary", cv=5, random_state=RANDOM_STATE),
         ALTA_CARDINALIDADE),
        ("vendedor", "passthrough", ["vendedor"]),
    ])), ("modelo", modelo)]), tabela_vazada_vendedor, y)
linhas.append({"método": "vendedor sem efeito, media calculada antes (vazado)", "colunas": 8,
               "roc_auc": media_ven, "desvio": desvio_ven})

comparacao = pd.DataFrame(linhas).set_index("método").round(4)
display(comparacao)


Leia a tabela em três pares, que é onde está a lição:

- **ordinal em tudo contra one-hot no plano**: 0,742 contra 0,775. Inventar ordem onde não existe custa
  caro, e é o erro mais comum de quem está começando, porque o código roda sem reclamar;
- **one-hot na cidade contra agrupar as raras**: 0,854 com 46 colunas contra 0,811 com 28. Aqui o
  one-hot ganhou, e isso também ensina: com trinta linhas por categoria o modelo dá conta, e agrupar
  joga fora o que havia. A terceira opção, codificar pelo alvo, chega a 0,849 com **6** colunas, que é
  o que se procura quando a cardinalidade é de milhares;
- **vendedor, a coluna sem efeito nenhum**: 0,847 no pipeline contra **0,951** com a média calculada
  antes da divisão. É o par que importa. Uma coluna que não tem relação alguma com o alvo melhora o
  resultado em dez pontos de AUC quando o encoding vê o alvo. Não é o modelo que ficou melhor, é o
  número que ficou falso.


## Valor faltante, nos dois tipos de coluna

Faltante em coluna numérica e faltante em coluna de texto são problemas parecidos com tratamentos
diferentes. E a imputação tem a mesma regra do encoding: se ela calcular a mediana antes da divisão,
essa mediana carrega informação do teste.


In [ ]:
imputadores = {
    "mediana (numerica) e mais frequente (texto)": (
        SimpleImputer(strategy="median"), SimpleImputer(strategy="most_frequent")),
    "KNN, 5 vizinhos": (KNNImputer(n_neighbors=5), SimpleImputer(strategy="most_frequent")),
    "iterativo": (IterativeImputer(random_state=RANDOM_STATE, max_iter=10),
                  SimpleImputer(strategy="most_frequent")),
}

faltantes = []
for nome, (imputa_num, imputa_cat) in imputadores.items():
    codigos = ColumnTransformer([
        ("num", Pipeline([("imputa", imputa_num), ("escala", StandardScaler())]), NUMERICAS),
        ("ordinal", Pipeline([("imputa", imputa_cat),
                              ("codifica", OrdinalEncoder(categories=[["baixo", "medio", "alto"]],
                                                          handle_unknown="use_encoded_value",
                                                          unknown_value=-1))]), ORDINAL),
        ("nominal", Pipeline([("imputa", imputa_cat),
                              ("codifica", OneHotEncoder(handle_unknown="ignore"))]), NOMINAIS),
        ("cidade", TargetEncoder(target_type="binary", cv=5, random_state=RANDOM_STATE),
         ALTA_CARDINALIDADE),
    ])
    media, desvio = auc_medio(Pipeline([("codigos", codigos), ("modelo", modelo)]), X, y)
    faltantes.append({"imputacao": nome, "roc_auc": round(media, 4), "desvio": round(desvio, 4)})

display(pd.DataFrame(faltantes).set_index("imputacao"))

# o faltante do texto pode virar categoria propria, em vez de ser imputado
plano_com_faltante = X["plano"].fillna("nao_informado")
print("com o faltante como categoria propria:",
      plano_com_faltante.value_counts().to_dict())
print(f"taxa de cancelamento de quem nao informou o plano: "
      f"{y[plano_com_faltante == 'nao_informado'].mean():.1%} "
      f"| de quem informou: {y[plano_com_faltante != 'nao_informado'].mean():.1%}")


Imputar rápido e simples costuma empatar com o método caro, e é isso que a tabela mostra: se a diferença
estiver dentro do desvio, o simples fica. O que **não** é opcional é que a imputação esteja dentro do
pipeline.

E o faltante do texto merece uma pergunta antes de ser imputado: quando o próprio "não informou" tem
relação com o alvo, ele é informação, não ausência, e merece ser uma categoria. Nesta tabela a diferença
aparece: 16,7% de cancelamento entre quem não informou o plano, contra 22,2% entre quem informou. Mudar
a direção do efeito de um mês para o outro é o normal, e é por isso que o caminho é olhar, não supor.


## O pipeline completo

Juntando tudo: um `ColumnTransformer` que trata cada tipo de coluna do jeito certo e um `Pipeline` que
liga isso ao modelo. O objeto resultante é o que vai para produção, porque ele carrega junto as
decisões de tratamento, e ele é o único jeito de garantir que a imputação e o encoding viram a partir
do treino.


In [ ]:
codigos = ColumnTransformer([
    ("num", Pipeline([("imputa", SimpleImputer(strategy="median")), ("escala", StandardScaler())]),
     NUMERICAS),
    ("ordinal", Pipeline([("imputa", SimpleImputer(strategy="most_frequent")),
                          ("codifica", OrdinalEncoder(categories=[["baixo", "medio", "alto"]],
                                                      handle_unknown="use_encoded_value",
                                                      unknown_value=-1))]), ORDINAL),
    ("nominal", Pipeline([("imputa", SimpleImputer(strategy="constant", fill_value="nao_informado")),
                          ("codifica", OneHotEncoder(handle_unknown="ignore"))]), NOMINAIS),
    ("cidade", TargetEncoder(target_type="binary", cv=5, random_state=RANDOM_STATE),
     ALTA_CARDINALIDADE),
])

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

for nome, classificador in {
    "regressao logistica": LogisticRegression(max_iter=5000),
    "gradient boosting em arvore": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}.items():
    t0 = time.perf_counter()
    pipeline = Pipeline([("codigos", codigos), ("modelo", classificador)]).fit(X_treino, y_treino)
    tempo = time.perf_counter() - t0
    auc_teste = roc_auc_score(y_teste, pipeline.predict_proba(X_teste)[:, 1])
    print(f"{nome:30s} roc_auc no teste: {auc_teste:.4f} | treino: {tempo:.1f}s")

print()
print("colunas depois do ColumnTransformer:",
      codigos.fit(X_treino, y_treino).get_feature_names_out().shape[0])
print("quais sao elas:")
print(list(codigos.get_feature_names_out()))


O `ColumnTransformer` é também o que permite dizer, em uma linha, que coluna recebe qual tratamento. É
o lugar onde a decisão do encoding deixa de ser escolha de quem escreveu o notebook e passa a ser
configuração que outra pessoa lê.

## Como ler isso

O roteiro, para qualquer tabela com categórica:

1. para cada coluna de texto, pergunte se a ordem existe;
2. conte as categorias: poucas pedem one-hot, muitas pedem agrupamento de raras ou codificação pelo
   alvo;
3. decisão que **não usa o alvo** (agrupar raras, juntar faltante numa categoria) pode ser tomada antes
   da divisão; decisão que usa o alvo, nunca;
4. deixe tudo dentro de um `ColumnTransformer` e de um `Pipeline`, e meça com validação cruzada;
5. antes de acreditar num número bom, pergunte se o tratamento viu o alvo.

O item 5 é o que separa um resultado de um engano, e é o tipo de erro que nunca aparece como erro.

## Resumo

Encoding é escolha medida, não sintaxe. Neste notebook: a ordem em `nivel_risco` entra como número, o
`plano` vira one-hot e o faltante dele vira categoria própria, a `cidade` troca 40 colunas por 1 número
calculado com validação cruzada interna, e o faltante da renda entra pela mediana. Imputar de forma
simples empatou com KNN e com imputação iterativa, o que é resultado comum e bom de ver.

E fica o número que vale mais que todos: `vendedor`, uma coluna sem efeito nenhum, faz o resultado ir
de 0,847 para 0,951 quando o encoding do alvo é calculado antes da divisão. O modelo não melhorou, a
medição é que ficou falsa.

**Próximo passo:** o notebook de engenharia de features desta pasta, onde a decisão deixa de ser "como
transformar a coluna que existe" e passa a ser "que coluna nova criar".

## Referências

```bibtex
@article{miccibarreca2001preprocessing,
  title   = {A Preprocessing Scheme for High-Cardinality Categorical Attributes in Classification
             and Prediction Problems},
  author  = {Micci-Barreca, Daniele},
  journal = {ACM SIGKDD Explorations Newsletter},
  volume  = {3},
  number  = {1},
  pages   = {27--32},
  year    = {2001},
  doi     = {10.1145/507533.507538}
}

@article{cerda2020encoding,
  title   = {Encoding High-Cardinality String Categorical Variables},
  author  = {Cerda, Patricio and Varoquaux, Ga{"e}l},
  journal = {IEEE Transactions on Knowledge and Data Engineering},
  volume  = {34},
  number  = {3},
  pages   = {1164--1176},
  year    = {2022},
  doi     = {10.1109/TKDE.2020.2992529}
}
```
